# Maps

*Rode uma célula com ▶ (ou Shift+Enter): o Sphinx compila e roda a célula no seu computador. Mude o código e rode de novo: este notebook é a sua cópia.*

**Depois desta unidade você consegue:**
- Guardar e buscar valores pela chave com um Map, contando e agrupando pelo caminho.
- Percorrer as entradas de um map, e escolher HashMap, LinkedHashMap, TreeMap ou EnumMap pela ordem de chaves de que precisa.

**Neste notebook**, uma ideia de cada vez (rode cada célula antes de seguir):
1. Map: buscando pela chave
2. Contando com um map
3. Ordem das chaves: Hash, Linked ou Tree
4. Um map de listas
5. EnumMap, e os seus tipos como chave
6. Qual usar?

Os **exercícios** deste notebook terminam com linhas `check(...)`: testes mínimos que imprimem ✓ quando o seu código está certo. Rode a próxima célula uma vez antes, para eles poderem usar o `check`.

In [ ]:
// shared: check(...) imprime ✓ ou ✗; os exercícios abaixo usam
void check(String what, Object actual, Object expected) {
    if (Objects.equals(actual, expected)) {
        IO.println("✓ " + what);
    } else {
        IO.println("✗ " + what + ": esperado " + expected + ", veio " + actual);
    }
}

## 1. Map: buscando pela chave

Um **Map** liga cada **chave** a um **valor**, como um dicionário liga uma palavra ao significado. `put(chave, valor)` guarda (e troca o valor antigo se a chave já existia), `get(chave)` lê (`null` quando a chave não existe), `getOrDefault(chave, x)` devolve `x` no lugar de `null`, e `containsKey` pergunta. Para percorrer, faça um laço em `entrySet()`: cada entrada tem `getKey()` e `getValue()`.

In [ ]:
Map<String, Double> prices = new HashMap<>();
prices.put("maçã", 0.5);
prices.put("pão", 2.25);
prices.put("maçã", 0.6);           // troca o 0.5
IO.println(prices.get("maçã"));
IO.println(prices.get("leite"));   // não está lá
IO.println(prices.getOrDefault("leite", 0.0));

List<String> cart = List.of("maçã", "pão", "maçã", "leite");
double total = 0;
for (String item : cart) {
    total += prices.getOrDefault(item, 0.0);
}
IO.println("Total: " + total);
for (Map.Entry<String, Double> e : prices.entrySet()) {
    IO.println(e.getKey() + " custa " + e.getValue());
}

🧪 **Experimente:** acrescente o leite na tabela de preços e rode de novo. Depois use `TreeMap` no lugar de `HashMap`: o que acontece com a ordem do último laço?

### ✏️ Exercício: Tamanhos das palavras

Monte um mapa de cada palavra para o tamanho dela. Uma palavra repetida só recebe a mesma entrada de novo.

In [ ]:
Map<String, Integer> lengths(List<String> words) {
    // TODO: ponha cada palavra com o tamanho dela
    return new HashMap<>();
}

void main() {
    check("duas palavras", lengths(List.of("hi", "there")), Map.of("hi", 2, "there", 5));
    check("repetida", lengths(List.of("a", "a")), Map.of("a", 1));
    check("vazio", lengths(List.of()), Map.of());
}

<details>
<summary>Ver uma resposta</summary>

```java
Map<String, Integer> lengths(List<String> words) {
    Map<String, Integer> result = new HashMap<>();
    for (String w : words) {
        result.put(w, w.length());
    }
    return result;
}

void main() {
    check("duas palavras", lengths(List.of("hi", "there")), Map.of("hi", 2, "there", 5));
    check("repetida", lengths(List.of("a", "a")), Map.of("a", 1));
    check("vazio", lengths(List.of()), Map.of());
}
```

</details>

## 2. Contando com um map

Uma tarefa muito comum: contar quantas vezes cada coisa aparece. Para cada item, some 1 à contagem dele, começando em 0 para uma chave nova. `merge(chave, 1, Integer::sum)` faz isso numa linha: põe 1 para uma chave nova, ou soma 1 ao valor antigo. Os slides acompanham isso.

#### 🎞️ Contando com um map

![merge acrescenta cada chave nova com 1 e soma 1 a uma chave que já existe; get devolve null para uma chave inexistente, enquanto getOrDefault devolve o padrão](map-steps.pt-br.svg)

In [ ]:
List<String> votes = List.of("vermelho", "azul", "vermelho", "verde", "vermelho", "azul");
Map<String, Integer> count = new HashMap<>();
for (String v : votes) {
    count.put(v, count.getOrDefault(v, 0) + 1);   // o jeito longo
}
IO.println(count);
Map<String, Integer> count2 = new HashMap<>();
for (String v : votes) {
    count2.merge(v, 1, Integer::sum);               // o jeito curto
}
IO.println(count2.equals(count));

🧪 **Experimente:** conte as letras de uma palavra em vez dos votos: percorra `word.toCharArray()` com um `Map<Character, Integer>`.

## 3. Ordem das chaves: Hash, Linked ou Tree

Como os sets, os maps vêm em três ordens. `HashMap` é o mais rápido, mas a ordem parece aleatória; `LinkedHashMap` mantém a ordem em que as chaves entraram; `TreeMap` mantém as chaves ordenadas e responde "a primeira chave", "as chaves antes de X".

In [ ]:
String[] countries = {"Peru", "Brasil", "Chile", "Angola"};
Map<String, Integer> hash = new HashMap<>();
Map<String, Integer> linked = new LinkedHashMap<>();
TreeMap<String, Integer> tree = new TreeMap<>();
for (String c : countries) {
    hash.put(c, c.length());
    linked.put(c, c.length());
    tree.put(c, c.length());
}
IO.println("Hash:   " + hash);
IO.println("Linked: " + linked);
IO.println("Tree:   " + tree);
IO.println("primeira: " + tree.firstKey() + ", antes de C: " + tree.headMap("C"));

🧪 **Experimente:** acrescente "Benin" aos países e veja onde ele fica em cada map. Depois imprima `tree.lastKey()`.

## 4. Um map de listas

Um valor pode ser uma coleção: `Map<String, List<String>>` dá a cada chave uma lista inteira. `computeIfAbsent(chave, k -> new ArrayList<>())` devolve a lista da chave, criando uma vazia na primeira vez, então dá para acrescentar nela na hora.

In [ ]:
String[][] classes = {{"Ana", "Java"}, {"Bia", "C"}, {"Caio", "Java"}, {"Davi", "Java"}};
Map<String, List<String>> students = new TreeMap<>();
for (String[] row : classes) {
    students.computeIfAbsent(row[1], k -> new ArrayList<>()).add(row[0]);
}
IO.println(students);
IO.println(students.get("Java").size() + " em Java");

🧪 **Experimente:** troque o `computeIfAbsent(...)` por `students.get(row[1]).add(row[0])` e rode: o primeiro `get` de cada turma é `null`.

### ✏️ Exercício: Palavras pela primeira letra

Monte um map de cada primeira letra para as palavras que começam com ela, na ordem original.

In [ ]:
Map<Character, List<String>> byFirstLetter(List<String> words) {
    // TODO: computeIfAbsent(w.charAt(0), …).add(w)
    return new HashMap<>();
}

void main() {
    check("três palavras", byFirstLetter(List.of("ant", "bee", "asp")), Map.of('a', List.of("ant", "asp"), 'b', List.of("bee")));
    check("nenhuma palavra", byFirstLetter(List.of()), Map.of());
}

<details>
<summary>Ver uma resposta</summary>

```java
Map<Character, List<String>> byFirstLetter(List<String> words) {
    Map<Character, List<String>> result = new HashMap<>();
    for (String w : words) {
        result.computeIfAbsent(w.charAt(0), k -> new ArrayList<>()).add(w);
    }
    return result;
}

void main() {
    check("três palavras", byFirstLetter(List.of("ant", "bee", "asp")), Map.of('a', List.of("ant", "asp"), 'b', List.of("bee")));
    check("nenhuma palavra", byFirstLetter(List.of()), Map.of());
}
```

</details>

## 5. EnumMap, e os seus tipos como chave

Quando as chaves são os valores de um enum, um **EnumMap** é a escolha natural: rápido, e sempre na ordem do enum. As suas classes também podem ser chaves, desde que `equals` e `hashCode` comparem os valores: records já fazem isso.

In [ ]:
enum Size { SMALL, MEDIUM, LARGE }

record Seat(char row, int number) {}

void main() {
    Map<Size, Integer> sold = new EnumMap<>(Size.class);
    sold.merge(Size.LARGE, 2, Integer::sum);
    sold.merge(Size.SMALL, 5, Integer::sum);
    sold.merge(Size.LARGE, 1, Integer::sum);
    IO.println(sold);                         // na ordem do enum

    Map<Seat, String> booked = new HashMap<>();
    booked.put(new Seat('A', 1), "Ana");
    IO.println(booked.get(new Seat('A', 1)));   // um record igual a encontra
}

🧪 **Experimente:** transforme o `record Seat` numa classe comum com construtor e rode: a busca dá `null`, porque a classe compara referências.

## 6. Qual usar?

| Você precisa de… | Use |
|---|---|
| itens em ordem, com repetidos, acesso pela posição | `List` (`ArrayList`) |
| cada item uma vez, "está lá?" rápido | `Set` (`HashSet`; `LinkedHashSet` para a ordem de chegada, `TreeSet` para ordenado) |
| um valor para cada chave | `Map` (`HashMap`; `LinkedHashMap`, `TreeMap`) |
| o primeiro a entrar sai primeiro, ou o último sai primeiro | `ArrayDeque` como fila ou pilha |

**Cuidado:** `map.get(chave)` devolve `null` para uma chave que não existe, e usar esse `null` como número para o programa com uma `NullPointerException`. Prefira `getOrDefault`.

In [ ]:
Map<String, Integer> stock = new HashMap<>();
stock.put("canetas", 12);
Integer pencils = stock.get("lapis");
IO.println(pencils);                              // null: chave inexistente
IO.println(stock.getOrDefault("lapis", 0) + 5);   // seguro

🧪 **Experimente:** escreva `int n = stock.get("lapis");` e rode: essa é a NullPointerException. Depois mude para usar `getOrDefault`.

## Em resumo

Um **Map** liga cada **chave** a um valor, como um dicionário liga uma palavra ao significado. Cada chave aparece uma vez: pô-la de novo troca o valor.

```java
void main() {
    Map<String, Integer> votes = new HashMap<>();
    for (String v : List.of("vermelho", "azul", "vermelho", "verde", "vermelho")) {
        votes.merge(v, 1, Integer::sum);       // soma 1, começando em 1
    }
    IO.println(votes.get("vermelho"));          // 3
    IO.println(votes.getOrDefault("rosa", 0));  // 0: chave inexistente

Map<String, Integer> sorted = new TreeMap<>(votes);
    for (Map.Entry<String, Integer> e : sorted.entrySet()) {
        IO.println(e.getKey() + ": " + e.getValue());
    }
}
```

`HashMap` não tem ordem útil de chaves, `LinkedHashMap` mantém a ordem de inserção, `TreeMap` mantém as chaves ordenadas, e `EnumMap` segue a ordem de um enum.

**Cuidado:** `get` devolve `null` para uma chave inexistente, e usar esse `null` como número quebra. Prefira `getOrDefault`.

## Leituras

- [Using Maps to Store Key Value Pairs](https://dev.java/learn/api/collections-and-streams/collections-framework/maps/) (dev.java (Oracle), 9 min): Como um map guarda pares chave/valor, depois "Practicing Map Operations": pegar um valor, getOrDefault, conferir uma chave e tirar uma entrada.

## Pratique

O notebook cobriu o básico. Agora aplique nos desafios da unidade, na barra lateral do Sphinx, onde o Submit roda todos os testes, inclusive os ocultos:

- **Frequência de palavras (HashMap)** · Médio
- **Agenda telefônica (TreeMap)** · Médio
- **Horas de estudo na semana (EnumMap)** · Médio
- **Palavras mais comuns** · Difícil